# Kiran — lab notebook

# Paper Review: Highly accurate protein structure prediction with AlphaFold

**Full citation:** Jumper, J., Evans, R., Pritzel, A., Green, T., Figurnov, M., Ronneberger, O., Tunyasuvunakool, K., Bates, R., Žídek, A., Potapenko, A., *et al.* (2021). Highly accurate protein structure prediction with AlphaFold. *Nature*, 596(7873), 583–589. https://doi.org/10.1038/s41586-021-03819-2

**Category:** AI/ML (applied to Biology/Biomedical: structural biology)

## 1. Summary
**What is this paper about? What problem does it address?**

- The paper presents **AlphaFold2**, a deep learning system that predicts a protein's 3D structure from its amino-acid sequence, often with near-experimental accuracy.
- It addresses the **protein folding problem**, which had been open for about 50 years. Experimental methods (X-ray crystallography, cryo-EM, NMR) are slow and expensive. Of the hundreds of millions of known protein sequences, only about 100k unique structures had been solved.
- Earlier computational methods (physics-based simulation, homology modeling, earlier deep learning such as AlphaFold1) fell well short of experimental accuracy, especially when no similar known structure existed.
- AlphaFold2 was the top entry by a wide margin in the blind **CASP14** assessment (2020). It is widely regarded as having largely solved single-chain structure prediction for many proteins.

## 2. Key Methods
**What data, model, or experimental approach did the authors use?**

**Inputs and data**
- The input is the target amino-acid sequence plus a **multiple sequence alignment (MSA)** of related sequences, searched from UniRef90, BFD and MGnify. Optional **structural templates** come from PDB70.
- Training used about 170k experimentally determined structures from the **Protein Data Bank (PDB)**.
- **Self-distillation:** the model predicted structures for about 350k diverse sequences (Uniclust30). The high-confidence predictions were added back as extra training data, which noticeably improved accuracy.

**Architecture**
- **Evoformer (48 blocks):** a new transformer-style trunk that jointly updates two representations: an MSA representation (sequences × residues) and a **pair representation** (residues × residues). Information flows between them through attention and "triangle" updates (triangle multiplicative updates and triangle self-attention). These updates push the pairwise distances toward geometric consistency, such as satisfying the triangle inequality.
- **Structure module (8 blocks):** treats each residue as a rigid body (a "residue gas" of rotations and translations). **Invariant Point Attention (IPA)** refines the 3D frames and predicts side-chain torsion angles, giving all-atom coordinates end to end.
- **Recycling:** outputs are fed back into the network three times to refine the prediction iteratively.

**Training objective and confidence**
- The main loss is **Frame Aligned Point Error (FAPE)**, with auxiliary losses: distogram, masked-MSA prediction (BERT-style), and structural violation terms.
- The model also predicts its own confidence. **pLDDT** is per-residue confidence on a 0–100 scale. **PAE (predicted aligned error)** is pairwise confidence about relative domain positions.

**Evaluation**
- Blind CASP14 targets, plus a larger post-training **"recent PDB" set** (about 10.8k chains released after the training cutoff) to show the results generalize.
- **Ablation studies** to measure how much each component contributes.

## 3. Main Results
**Key findings, important figures or tables, specific metrics**

- **CASP14 accuracy (Fig. 1):** median backbone accuracy of **0.96 Å RMSD₉₅** (Cα). The next-best method scored **2.8 Å**. All-atom accuracy was **1.5 Å RMSD₉₅** versus 3.5 Å for the next best. For scale, a carbon atom is about 1.4 Å wide.
- **Side chains:** when the backbone was predicted well, side-chain positions were also highly accurate.
- **Generalization (Fig. 2):** accuracy stayed high on the recent-PDB set, including chains with no close templates. Very large proteins were also handled well.
- **Calibrated confidence:** pLDDT correlates strongly with true accuracy (lDDT-Cα), and PAE reliably flags uncertain relative positions of domains. Users can therefore tell which predictions to trust.
- **Ablations (Fig. 4):** removing the triangle updates, IPA, recycling, self-distillation or masked-MSA training each reduced accuracy. The Evoformer pair representation and self-distillation were among the biggest contributors.
- **MSA depth (Fig. 5):** accuracy drops sharply when the alignment is shallow (median per-residue Neff below about 30 sequences). The model depends heavily on evolutionary/coevolutionary signal.
- **Limitations:** weaker on proteins with few homologs and on intrinsically disordered regions (which receive low pLDDT). The model is designed for single chains, so protein complexes, conformational changes, and effects of ligands/mutations are not directly modeled; AlphaFold-Multimer and AlphaFold3 were developed later to cover more of this.
- **Impact:** led to the **AlphaFold Protein Structure Database** (now 200M+ predicted structures) and open-source code. Recognized in the **2024 Nobel Prize in Chemistry** (Hassabis and Jumper).

## 4. Connection to Our Project
**How does this relate to what our team is doing? What can we borrow, build on, or do differently?**

**How it relates**
- Our project tests the usual explanation for why AlphaFold2 barely changes its prediction after a single substitution: one changed residue is outweighed by thousands of homologs in the MSA. This paper is where that dependence on the alignment comes from. AlphaFold2 is built around the MSA, and the Evoformer reads coevolutionary signal from it into the pair representation.
- The paper's own **MSA-depth analysis (Fig. 5)** shows accuracy falling sharply once the alignment is shallow (median Neff below about 30). That is the most direct published evidence that alignment depth controls AlphaFold2's behavior. It supports H1 and is the basis of our planned subsampling **extension**.
- The paper evaluates accuracy only against wild-type experimental structures. It never asks whether the model responds to sequence variation, so the mutation question we're asking falls outside what it was designed or validated for.

**What we can borrow**
- **Evaluation design:** the paper reports **per-residue** accuracy and confidence (lDDT, pLDDT) alongside global measures. This matches our decision to treat per-residue displacement as the main result and global RMSD only as a point of comparison with prior work. lDDT doesn't require superposition, so it also gives us a superposition-free check on our displacement measurements.
- **Confidence scores:** pLDDT is trained to predict lDDT, a measure of local accuracy, which is why we treat a change in confidence as a statement about **certainty, not energy**. PAE could add an inter-residue view of where uncertainty moves after a substitution.
- **Accuracy floor:** the paper's reported accuracy against experimental structures shows how close a predicted wild type should sit to reality. Our wild-type vs. experimental comparison should land in a similar range for these well-studied proteins. If it doesn't, check the setup before interpreting any mutant effects.
- **Ablation logic:** the paper removes one component at a time to measure how much each contributes. Our ColabFold vs. ESMFold comparison does the same thing with the alignment, and the subsampling extension turns it into a graded version.

**Cautions for our design**
- **Noise floor:** with fixed weights and inputs, inference may be close to deterministic. "Replicate folds" therefore need a defined source of variation, such as ColabFold's five model weights, random seeds or dropout. For ESMFold, check whether repeated runs differ at all. A noise floor of exactly zero would make every displacement look significant, so state what varies between replicates.
- **Memorization:** ubiquitin, PDZ3, avGFP and TEM-1 all have many structures in the PDB, which both predictors were trained on. Some insensitivity may come from recalling the wild-type fold rather than from the alignment, which would argue toward H0 rather than H1. Turning templates off in ColabFold separates template use from MSA use. Training-set overlap remains a limitation to state.
- **Single static structure:** AlphaFold2 predicts one structure and has no energy term, so it doesn't model stability directly. A weak link between displacement and fitness (H2) may reflect this limit rather than noise alone. The experimental mutant structures act as our ground truth for how much the real proteins actually move.

**What we do differently**
- We test the model where it is weakest: variation **within one protein family** rather than differences between folds. We add the controls the existing literature lacks: run-to-run noise, distance from the experimental structure, the substituted wild-type structure with no repacking, and matched experimental pairs.
- Removing the alignment entirely (ESMFold) directly tests the alignment explanation. Neither this paper nor the later studies of AlphaFold2 and mutations do this.